In [27]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

from pathlib import Path
from langchain_community.document_loaders import TextLoader,PyPDFLoader, Docx2txtLoader

load_dotenv()
# print("Ключ найден:", bool(os.getenv("GEMINI_API_KEY")))

import warnings
warnings.filterwarnings("ignore")

In [16]:
def load_documents(folder="data"):
    docs = []
    for path in sorted(Path(folder).iterdir()):
        ext = path.suffix.lower()
        if ext == ".txt":
            loaded = TextLoader(str(path), encoding="utf-8").load()
        elif ext == ".pdf":
            loaded = PyPDFLoader(str(path)).load()
        elif ext == ".docx":
            loaded = Docx2txtLoader(str(path)).load()
        else:
            continue
        for d in loaded:
            d.metadata["source"] = path.name
        docs += loaded
        print(f"{path.name}: {len(loaded)} часть(ей)")
    return docs

documents = load_documents()
print("Всего частей:", len(documents))

expense_policy.txt: 1 часть(ей)
flexible_pto.docx: 1 часть(ей)
public_holidays.pdf: 2 часть(ей)
support_time_off.txt: 1 часть(ей)
Всего частей: 5


In [17]:
for d in documents:
    print(d.metadata["source"], "-", len(d.page_content.split()), "слов")

expense_policy.txt - 199 слов
flexible_pto.docx - 594 слов
public_holidays.pdf - 449 слов
public_holidays.pdf - 126 слов
support_time_off.txt - 3015 слов


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
# gemini-3.8-flash 
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0,api_key=os.getenv("GEMINI_API_KEY"),)
print(llm.invoke("Say: Hi").content)

# разбиение на куски

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

spliter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
splits = spliter.split_documents(documents)
print("Parts:", len(splits))

Parts: 50


# embedding and DB Chroma

In [20]:
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embelding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=os.getenv("GEMINI_API_KEY"),)
vectordb = Chroma(persist_directory="chroma_db", embedding_function=embelding)
in_db = {m["source"] for m in vectordb.get(include=["metadatas"])["metadatas"]}
new_splits = [s for s in splits if s.metadata["source"]not in in_db]

if new_splits:
    vectordb.add_documents(new_splits)
    print("Добавлено кусков: ", len(new_splits))
    print("Из файлов: ", sorted({s.metadata["source"] for s in new_splits}))
else:
    print("Новых файлов нет")
print("in database: ", vectordb._collection.count())

Новых файлов нет
in database:  50


# проверка поиска (без LLM)

In [21]:
results = vectordb.similarity_search("How many days of vacation can I take in a row?", k=3)
for r in results:
    print(r.metadata["source"], "|", r.page_content[:150].replace("\n"," "))
    print("---")

flexible_pto.docx | We strongly recommend taking at least two consecutive weeks of time off and at minimum 25 days total throughout the year.  PTO Protocols (Planning and
---
flexible_pto.docx | Our Flexible PTO policy means that as long as you have coordinated with your manager about your planned time off and made a coverage plan, you are fre
---
public_holidays.pdf | to   3   days.     I   still   have   questions   related   to   public   holidays   and/or   how   to   categorize   my   time   off,   who   can   I
---


# Ячейка: цепочка

In [22]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""Ты помощник по внутренним документам компании.
Отвечай только на основе контекста ниже, ничего не выдумывай.
Если в контексте есть частично подходящая информация, дай её и объясни, чего именно не хватает.
Только если в контексте вообще нет ничего по теме вопроса, скажи:
"В документах нет ответа на этот вопрос."
Отвечай на том языке, на котором задан вопрос.

Контекст:
{context}

Вопрос: {question}
Ответ: """)
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

llm_answer = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    api_key=os.getenv("GEMINI_API_KEY"),
    max_retries=1,
    timeout=30,
)
llm_condense = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
    api_key=os.getenv("GEMINI_API_KEY"),
    max_retries=1,
    timeout=30,
)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm_answer,
    condense_question_llm=llm_condense,
    retriever=vectordb.as_retriever(search_kwargs={"k": 4}),
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": prompt},
)

In [23]:
import time

def safe_invoke(question, tries=4, pause=5):
    for attempt in range(1, tries + 1):
        try:
            return qa.invoke({"question": question})
        except Exception as e:
            print(f"Попытка {attempt} не удалась: {str(e)[:120]}")
            if attempt == tries:
                raise
            time.sleep(pause * attempt)

# Ячейка: функция вопроса

In [24]:
try:
    v = embelding.embed_query("test")
    print("Работает, длина вектора:", len(v))
except Exception as e:
    print("Ошибка:", str(e)[:300])

Работает, длина вектора: 3072


In [25]:
import panel as pn
pn.extension(theme="default")

# def respond(contents, user, instance):
#     result = qa.invoke({"question": contents})
#     sources = sorted({d.metadata["source"] for d in result["source_documents"]})
#     return f"{result['answer']}\n\n**Источники:** {', '.join(sources)}"
def respond(contents, user, instance):
    result = safe_invoke(contents)
    sources = sorted({d.metadata["source"] for d in result["source_documents"]})
    text = f"{result['answer']}\n\n**Источники:** {', '.join(sources)}"
    return pn.pane.Markdown(text)

chat = pn.chat.ChatInterface(
    callback=respond,
    user="User",
    callback_user="AI Assistant",
    message_params={
        "stylesheets": [
            ":host .message { color: #000000 !important; background-color: #F6F6F6 !important; }"
        ]
    },
)
chat.send("Здравствуйте! Задайте вопрос по документам GitLab Handbook", user="AI Assistant", respond=False)
chat

BokehModel(combine_events=True, render_bundle={'docs_json': {'c4b7c3fc-9ce3-4dac-9a77-e8627c1f2b9d': {'version…

# проверка качества

In [26]:
import json

with open("eval/questions.json", encoding="utf-8") as f:
    questions = json.load(f)

for i, q in enumerate(questions, 1):
    memory.clear() 
    result = qa.invoke({"question": q["question"]})
    sources = sorted({d.metadata["source"] for d in result["source_documents"]})
    print(f"{i}. {q['question']}")
    print("   Ответ бота:", result["answer"])
    print("   Ожидалось:", q["expected"])
    print("   Источники:", sources, "| нужный:", q["source"])
    print()

1. I joined with 92 days remaining in the year and the country's full-year public holiday entitlement is 10 days. What is my balance?
   Ответ бота: Ваш баланс составляет 3 дня.

Согласно документам, расчет производится по формуле: (Количество оставшихся дней в календарном году ÷ 365) × Годовая норма государственных праздников.

В вашем случае: 10 × (92 ÷ 365) = 2,52. Результат округляется до ближайшего целого числа (в большую сторону, если десятичная часть составляет .5 или выше), что дает 3 дня.
   Ожидалось: 3 days (2.52 rounded up)
   Источники: ['public_holidays.pdf'] | нужный: public_holidays.pdf

2. Can I take part of a public holiday, for example half a day?
   Ответ бота: Нет, вы не можете взять часть государственного праздника. В предоставленном документе прямо указано: "You cannot take partial public holidays."
   Ожидалось: ЗАПОЛНИТЬ из public_holidays.pdf
   Источники: ['public_holidays.pdf'] | нужный: public_holidays.pdf

3. Сколько дней подряд можно взять в рамках Flexib